In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # project root, so `src` imports work
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
from src import config

# 04 — Modeling & Evaluation (Deliverable D)

All scores come from chronological splits of the training file — never the test file.
**Main split:** train on 1 Jan – 17 Oct 2025, validate on 18 – 31 Oct 2025 (14 days, the same horizon as the
forecast). **Rolling origin:** 5 folds, each trained on everything before the cut and validated on the next 14 days.
Fitted features (zone types, zone x weekday x hour profile) are re-learned inside every fold from that fold's
training rows only (`src/train.refit_fold`). Writes `reports/D_model_evaluation.md`, `models/final_model.joblib`
and figures 10–12.

In [2]:
import time, joblib
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance
from sklearn.model_selection import KFold
from src import train as T, features, plotstyle as S
from src.nbtools import Report
S.apply()
m = T.load_master_train()
MAIN_CUT = config.VALID_START[:10]
CUTS = ["2025-08-23", "2025-09-06", "2025-09-20", "2025-10-04", "2025-10-18"]
TUNE_CUTS = CUTS[:-1]            # tuning never sees the main validation fortnight
BASE = features.BASE_FEATURES
FINAL = features.ALL_FEATURES
tr, va = T.split(m, MAIN_CUT)
R = Report("D — Modeling & Evaluation (team " + config.TEAM_NAME + ")")
R.p(f"Generated by `notebooks/04_modeling_and_evaluation.ipynb`. **Main split:** train {tr.pickup_hour.min():%d %b} – "
    f"{tr.pickup_hour.max():%d %b %Y} ({len(tr):,} zone-hours), validate {va.pickup_hour.min():%d %b} – {va.pickup_hour.max():%d %b %Y} "
    f"({len(va):,} zone-hours, mean {va.trips.mean():.1f} trips). **Rolling origin:** cuts " + ", ".join(CUTS) +
    ", each validating the next 14 days. Only observed zone-hours are trained on and scored. Fitted features are re-learned "
    "inside each fold from its training rows only. RMSE and MAE are in trips per zone-hour.")
print(len(tr), len(va))

79190 3914


## D1 — Baselines

In [3]:
base_rows = []
for name, fn in [("mean predictor", T.mean_baseline),
                 ("seasonal naive (zone x weekday x hour, whole training period)", T.seasonal_naive),
                 ("seasonal naive (zone x weekday x hour, last 4 weeks)", lambda a, b: T.seasonal_naive(a, b, 4)),
                 ("moving average (zone x hour, last 7 days)", T.moving_average)]:
    p = fn(tr, va)
    base_rows.append({"model": name, "rmse": T.rmse(va.trips, p), "mae": T.mae(va.trips, p)})
d1 = pd.DataFrame(base_rows)
SN = d1.loc[2, "rmse"]
R.h("D1 Baselines")
R.table(d1, floatfmt=".2f")
R.p(f"The mean predictor ignores zone and hour and is useless (RMSE {d1.loc[0,'rmse']:.1f}). The seasonal-naive profile cuts the error "
    f"by more than half, and using only the **last 4 weeks** is better again ({d1.loc[2,'rmse']:.2f} vs {d1.loc[1,'rmse']:.2f}) because demand "
    "grows through the year (B1.4) and the full-period average is stale. The 7-day **moving average** is recent but blind to the "
    f"weekday ({d1.loc[3,'rmse']:.2f}): weekends look nothing like weekdays in the business zones, so averaging them together costs more "
    "than the freshness gains. The 4-week seasonal naive is the bar every model must beat.")
d1

,model,rmse,mae
0,mean predictor,27.531086,20.689133
1,"seasonal naive (zone x weekday x hour, whole t...",11.298446,7.360085
2,"seasonal naive (zone x weekday x hour, last 4 ...",9.943454,6.636410
3,"moving average (zone x hour, last 7 days)",14.548444,9.491496


## D2 — Model comparison

In [4]:
d2_rows, PRED_MAIN, ROLL = [], {}, {}
labels = {"arima": "ARIMA(2,0,1) on daily/weekly Fourier seasonality, one per zone (time series only)",
          "ridge": "Ridge (linear, log target, one-hot zone/hour/weekday)", "random_forest": "Random forest (log target)",
          "gradient_boosting": "Gradient boosting, scikit-learn classic (log target)",
          "hist_gb": "HistGradientBoosting (log target)", "lightgbm": "LightGBM (Poisson objective)"}
def predictor(name):
    if name == "arima":
        return T.arima_forecast
    return lambda a, b: T.fit_predict(name, a, b, BASE)[1]
for name in labels:
    t0 = time.perf_counter(); p = predictor(name)(tr, va); secs = time.perf_counter() - t0
    PRED_MAIN[name] = p
    ROLL[name] = T.rolling_origin(m, CUTS, predictor(name))
    d2_rows.append({"model": labels[name], "rmse": T.rmse(va.trips, p), "mae": T.mae(va.trips, p),
                    "rolling_rmse_mean": ROLL[name].rmse.mean(), "fit_seconds": secs,
                    "rmse_vs_seasonal_naive_pct": 100 * (T.rmse(va.trips, p) / SN - 1)})
ROLL["moving_average"] = T.rolling_origin(m, CUTS, T.moving_average)
roll_sn_mean = T.rolling_origin(m, CUTS, lambda a, b: T.seasonal_naive(a, b, 4)).rmse.mean()
d2 = pd.DataFrame(d2_rows).sort_values("rmse")
R.h("D2 Model comparison")
R.p(f"Six model families on the same split. The machine-learning models all get the same {len(BASE)} features (the D8 feature is "
    "added later). ARIMA is the classical time-series model: one per zone, fitted on that zone's last 8 weeks of hourly trips "
    "(log scale) as daily and weekly Fourier seasonality plus ARIMA(2,0,1) on the remainder, forecasting the whole 14 days "
    "ahead; it sees no weather or events. `rolling_rmse_mean` is the mean over the 5 rolling-origin folds (D3); `fit_seconds` is one fit plus prediction.")
R.table(d2, floatfmt=".2f")
win = d2.iloc[0]
R.p(f"**Winner: {win.model}** (RMSE {win.rmse:.2f}, {abs(win.rmse_vs_seasonal_naive_pct):.0f}% better than the 4-week seasonal naive). "
    f"The three gradient-boosting models (LightGBM {d2.set_index('model').loc[labels['lightgbm'],'rmse']:.2f}, HistGradientBoosting "
    f"{d2.set_index('model').loc[labels['hist_gb'],'rmse']:.2f}, classic gradient boosting "
    f"{d2.set_index('model').loc[labels['gradient_boosting'],'rmse']:.2f}) lead, ahead of the random forest and the linear model: demand is "
    "driven by interactions (zone x hour x weekday, rain x zone type, event phase x zone) that trees find automatically and a linear "
    f"model cannot. **ARIMA** ({d2.set_index('model').loc[labels['arima'],'rmse']:.2f}; rolling "
    f"{ROLL['arima'].rmse.mean():.2f} vs seasonal naive {roll_sn_mean:.2f}) does no better than the seasonal naive: over a "
    "14-day horizon its short-memory ARIMA part fades to the smooth seasonal curve within hours, and it cannot see rain, events or "
    "holidays — the features that explain the departures from the usual weekly shape. We pick LightGBM because it is the most accurate, "
    "trains in seconds, handles missing lags natively and its Poisson objective matches count data.")
d2

,model,rmse,mae,rolling_rmse_mean,fit_seconds,rmse_vs_seasonal_naive_pct
5,LightGBM (Poisson objective),8.069950,5.584440,8.773447,4.041431,-18.841582
4,HistGradientBoosting (log target),8.200110,5.647619,8.989428,1.940776,-17.532580
3,"Gradient boosting, scikit-learn classic (log t...",8.630497,5.923848,9.161743,91.308651,-13.204234
2,Random forest (log target),8.786875,5.972267,9.597797,28.817268,-11.631563
0,"ARIMA(2,0,1) on daily/weekly Fourier seasonali...",10.412329,6.863548,11.519157,4.972630,4.715413
1,"Ridge (linear, log target, one-hot zone/hour/w...",12.987429,7.796316,13.945100,0.686032,30.612856


## D3 — Rolling-origin validation

In [5]:
roll_lgb = ROLL["lightgbm"]
roll_sn = T.rolling_origin(m, CUTS, lambda a, b: T.seasonal_naive(a, b, 4))
d3 = roll_lgb[["cut", "valid_to", "n_train", "n_valid", "mean_trips"]].copy()
d3["lightgbm_rmse"] = roll_lgb.rmse; d3["lightgbm_mae"] = roll_lgb.mae
d3["seasonal_naive_rmse"] = roll_sn.rmse; d3["improvement_pct"] = 100 * (1 - d3.lightgbm_rmse / d3.seasonal_naive_rmse)
worst = d3.loc[d3.lightgbm_rmse.idxmax()]
R.h("D3 Rolling-origin validation")
R.table(d3, floatfmt=".2f")
R.p(f"LightGBM: RMSE **{roll_lgb.rmse.mean():.2f} ± {roll_lgb.rmse.std():.2f}** (mean ± sd over {len(CUTS)} folds); seasonal naive "
    f"{roll_sn.rmse.mean():.2f} ± {roll_sn.rmse.std():.2f}. The model wins in every fold, and its spread is small relative to the gap, so "
    f"the main-split result is not luck. The worst fold is {worst.cut} – {worst.valid_to} (RMSE {worst.lightgbm_rmse:.2f}): it contains "
    "Meskel eve (26 Sep, the Demera bonfire at Meskel Square next to Kazanchis) and the end of the rainy season, and the Ethiopian "
    "New Year eve falls in the fold before — unlisted holiday-eve crowds the features cannot see (D7).")
d3

,cut,valid_to,n_train,n_valid,mean_trips,lightgbm_rmse,lightgbm_mae,seasonal_naive_rmse,improvement_pct
0,2025-08-23,2025-09-05,63462,3937,31.414275,8.237914,5.690553,11.312403,27.178036
1,2025-09-06,2025-09-19,67399,3930,32.105725,9.450056,5.953058,12.051590,21.586641
2,2025-09-20,2025-10-03,71329,3926,33.873281,9.991443,6.079630,13.447377,25.699684
3,2025-10-04,2025-10-17,75255,3935,32.001398,8.117869,5.434243,10.654276,23.806470
4,2025-10-18,2025-10-31,79190,3914,33.447752,8.069950,5.584440,9.943454,18.841582


## D4 — Feature availability & leakage audit

In [6]:
cand = []
for f in FINAL:
    fam = next(g for g, fs in [("calendar", features.CALENDAR_FEATURES + features.D8_FEATURES), ("zone", features.ZONE_FEATURES),
                               ("weather", features.WEATHER_FEATURES), ("events", features.EVENT_FEATURES),
                               ("lag / trend", features.LAG_FEATURES)] if f in fs)
    note = {"weather": "weather forecast for the fortnight (data_type = forecast)",
            "lag / trend": ">= 14 days old, so known on 31 Oct for every hour to 14 Nov"}.get(fam, "known from the calendar / event schedule")
    cand.append({"column": f, "family": fam, "known_at_forecast_time": "yes", "used": "yes", "why": note})
for f, why in [("active_drivers", "only observed after the hour happens; drivers follow demand (r = 0.99)"),
               ("avg_wait_min", "consequence of demand vs supply in that hour"),
               ("avg_fare_birr", "only observed after the trips happen"),
               ("lag_1h / lag_24h (not built)", "the hour before is unknown for most of the 14-day horizon"),
               ("ev_cancelled_window", "cancelled events did not happen; kept for analysis only (B3.4)"),
               ("row_status / trips_spike_fixed", "describe the training record, not the future hour")]:
    cand.append({"column": f, "family": "excluded", "known_at_forecast_time": "no", "used": "no", "why": why})
d4 = pd.DataFrame(cand)
leak_feats = BASE + ["active_drivers", "avg_wait_min"]
trL = tr.dropna(subset=["avg_wait_min"]); vaL = va.dropna(subset=["avg_wait_min"])
_, pL, _ = T.fit_predict("lightgbm", trL, vaL, leak_feats)
_, pB, _ = T.fit_predict("lightgbm", trL, vaL, BASE)
# Random-split contrast (shuffled 5-fold on the training rows) - shown only to illustrate optimism.
kf = KFold(5, shuffle=True, random_state=config.RANDOM_STATE)
rs = []
for a, b in kf.split(tr):
    mdl = T.make_model("lightgbm", BASE).fit(tr.iloc[a][BASE], tr.iloc[a].trips)
    rs.append(T.rmse(tr.iloc[b].trips, mdl.predict(tr.iloc[b][BASE])))
d4b = pd.DataFrame([
    {"experiment": "honest model (forecast-time features)", "validation": "chronological, 18-31 Oct", "rmse": T.rmse(vaL.trips, pB)},
    {"experiment": "+ active_drivers + avg_wait_min (LEAKY)", "validation": "chronological, 18-31 Oct", "rmse": T.rmse(vaL.trips, pL)},
    {"experiment": "honest model", "validation": "random shuffled 5-fold (contrast only)", "rmse": float(np.mean(rs))},
])
R.h("D4 Feature availability & leakage audit")
R.table(d4)
R.table(d4b, floatfmt=".2f")
R.p(f"Adding `active_drivers` and `avg_wait_min` drops the validation RMSE from {d4b.rmse[0]:.2f} to **{d4b.rmse[1]:.2f}** — it looks like a "
    "huge win, but it is leakage: those columns are measured during the hour we are trying to forecast (drivers follow demand at 1.3 "
    "trips each), so in production next Tuesday's driver count does not exist and the model would have nothing to read. Likewise a "
    f"shuffled random split gives an optimistic {d4b.rmse[2]:.2f} because neighbouring hours of the same days land on both sides; "
    "it is shown only as a contrast and never used for selection.")
d4b

,experiment,validation,rmse
0,honest model (forecast-time features),"chronological, 18-31 Oct",8.104536
1,+ active_drivers + avg_wait_min (LEAKY),"chronological, 18-31 Oct",3.153361
2,honest model,random shuffled 5-fold (contrast only),7.227399


## D5 — Ablation

In [7]:
abl = []
for name, feats in T.FEATURE_SETS.items():
    r = T.rolling_origin(m, CUTS, lambda a, b, f=feats: T.fit_predict("lightgbm", a, b, f)[1])
    _, p, _ = T.fit_predict("lightgbm", tr, va, feats)
    abl.append({"feature set": name, "n_features": len(feats), "main_split_rmse": T.rmse(va.trips, p), "main_split_mae": T.mae(va.trips, p),
                "rolling_rmse_mean": r.rmse.mean(), "rolling_mae_mean": r.mae.mean()})
d5 = pd.DataFrame(abl)
ref = d5.loc[0]
d5["delta_rolling_rmse_vs_calendar"] = d5.rolling_rmse_mean - ref.rolling_rmse_mean
d5["delta_rolling_mae_vs_calendar"] = d5.rolling_mae_mean - ref.rolling_mae_mean
R.h("D5 Ablation")
R.p("Final model type (LightGBM, Poisson) on identical splits; 'calendar+zone+trend' includes the holiday/school-break calendar "
    "and the lag/trend features; '+events' adds the venue-event features; '+weather' adds the six weather features.")
R.table(d5, floatfmt=".3f")
gw = -d5.loc[1, "delta_rolling_rmse_vs_calendar"]; ge = -d5.loc[2, "delta_rolling_rmse_vs_calendar"]; gb = -d5.loc[3, "delta_rolling_rmse_vs_calendar"]
R.p(f"Both joins pay off. Weather lowers the rolling RMSE by **{gw:.2f}** ({100*gw/ref.rolling_rmse_mean:.1f}%) and events by **{ge:.2f}** "
    f"({100*ge/ref.rolling_rmse_mean:.1f}%); together {gb:.2f} ({100*gb/ref.rolling_rmse_mean:.1f}%). Weather likely helps more than events because "
    "rain touches many hours across all zones, while events touch only a few zones for a few hours each.")
d5

,feature set,n_features,main_split_rmse,main_split_mae,rolling_rmse_mean,rolling_mae_mean,delta_rolling_rmse_vs_calendar,delta_rolling_mae_vs_calendar
0,calendar+zone+trend,17,8.775236,5.891813,9.934425,6.325173,0.000000,0.000000
1,+weather,23,8.275569,5.687018,9.055205,5.846757,-0.879220,-0.478416
2,+events,29,8.634406,5.825099,9.584665,6.222182,-0.349759,-0.102991
3,+both,35,8.069950,5.584440,8.773447,5.748385,-1.160978,-0.576788


## D6 — Hyperparameter tuning

In [8]:
SPACE = {"num_leaves": [15, 31, 63, 127], "learning_rate": [0.02, 0.03, 0.05, 0.08], "n_estimators": [400, 700, 1000, 1400],
         "min_child_samples": [10, 20, 40, 80], "colsample_bytree": [0.5, 0.7, 0.9], "subsample": [0.6, 0.8, 1.0],
         "reg_lambda": [0.0, 1.0, 5.0, 20.0]}
N_TRIALS = 16
rng = np.random.default_rng(config.RANDOM_STATE)
folds = [T.split(m, c) for c in TUNE_CUTS]
def cv_score(params):
    return float(np.mean([T.rmse(b.trips, T.fit_predict("lightgbm", a, b, BASE, params)[1]) for a, b in folds]))
trials = [{"trial": 0, **{k: T.LGB_PARAMS[k] for k in SPACE}, "cv_rmse": cv_score({})}]
for i in range(1, N_TRIALS + 1):
    p = {k: v[rng.integers(len(v))] for k, v in SPACE.items()}
    p = {k: (int(v) if isinstance(v, (np.integer,)) else float(v) if isinstance(v, np.floating) else v) for k, v in p.items()}
    trials.append({"trial": i, **p, "cv_rmse": cv_score(p)})
d6 = pd.DataFrame(trials).sort_values("cv_rmse")
best = {k: d6.iloc[0][k] for k in SPACE}
best = {k: (int(v) if k in ("num_leaves", "n_estimators", "min_child_samples") else float(v)) for k, v in best.items()}
_, p_def, _ = T.fit_predict("lightgbm", tr, va, BASE)
_, p_tun, _ = T.fit_predict("lightgbm", tr, va, BASE, best)
roll_tun = T.rolling_origin(m, CUTS, lambda a, b: T.fit_predict("lightgbm", a, b, BASE, best)[1])
d6s = pd.DataFrame([
    {"params": "default (trial 0)", "tuning_cv_rmse": d6[d6.trial == 0].cv_rmse.iloc[0], "main_split_rmse": T.rmse(va.trips, p_def),
     "main_split_mae": T.mae(va.trips, p_def), "rolling_rmse_mean": roll_lgb.rmse.mean()},
    {"params": "tuned (best trial)", "tuning_cv_rmse": d6.cv_rmse.iloc[0], "main_split_rmse": T.rmse(va.trips, p_tun),
     "main_split_mae": T.mae(va.trips, p_tun), "rolling_rmse_mean": roll_tun.rmse.mean()}])
R.h("D6 Hyperparameter tuning")
R.p(f"Random search, {N_TRIALS} trials + the default, scored by mean RMSE over the 4 earlier rolling folds ({', '.join(TUNE_CUTS)}) "
    "so the main validation fortnight stays untouched. Search space: " + "; ".join(f"`{k}` {v}" for k, v in SPACE.items()) + ".")
R.p("All trials, best first:")
R.table(d6, floatfmt=".3f")
R.p("Best parameters: " + ", ".join(f"`{k}`={v}" for k, v in best.items()) + ".")
R.table(d6s, floatfmt=".3f")
R.p(f"Tuning moves the main-split RMSE from {T.rmse(va.trips, p_def):.2f} to {T.rmse(va.trips, p_tun):.2f} and the rolling mean from "
    f"{roll_lgb.rmse.mean():.2f} to {roll_tun.rmse.mean():.2f}. The tuned settings win on the 4 folds they were chosen on and on the "
    "5-fold average, but are slightly worse on the main fortnight alone: the difference is within fold-to-fold noise. We keep the tuned "
    "settings because they were selected without looking at the main fortnight (a smaller, faster model); the features matter far "
    "more than the knobs.")
d6s

,params,tuning_cv_rmse,main_split_rmse,main_split_mae,rolling_rmse_mean
0,default (trial 0),8.949321,8.069950,5.584440,8.773447
1,tuned (best trial),8.881429,8.137937,5.610806,8.732730


## D7 — Error analysis

In [9]:
from src import analysis as A
ev = A.load_events()
hol = ev[(ev.event_type == "public_holiday") & (ev.status == "confirmed")]
eve_name = {(d - pd.Timedelta(days=1)).date(): n for d, n in zip(hol.start.dt.floor("D"), hol.event_name)}
oof = []
for c in CUTS:
    a, b = T.split(m, c)
    oof.append(b.assign(pred=T.fit_predict("lightgbm", a, b, BASE, best)[1], fold=c))
oof = pd.concat(oof, ignore_index=True)
oof["err"] = oof.trips - oof.pred; oof["abs_err"] = oof.err.abs()
oof["day_type"] = np.select([oof.is_public_holiday == 1, oof.is_weekend == 1], ["holiday", "weekend"], "weekday")
def summ(g):
    return pd.Series({"zone_hours": len(g), "mean_trips": g.trips.mean(), "rmse": np.sqrt((g.err ** 2).mean()),
                      "mae": g.abs_err.mean(), "bias(actual-pred)": g.err.mean()})
by_zone = oof.groupby("zone").apply(summ).sort_values("rmse", ascending=False)
by_zone["rmse_pct_of_mean"] = 100 * by_zone.rmse / by_zone.mean_trips
by_hour = oof.groupby("hour").apply(summ)
by_day = oof.groupby("day_type").apply(summ)
venue_types = set(ev[ev.event_type.isin(features.VENUE_EVENT_TYPES)].event_id)
def hypothesis(r):
    d = r.pickup_hour.date()
    ids = [x for x in str(r.ev_ids).split(";") if x in venue_types]
    if d in eve_name:
        return f"eve of {eve_name[d]}: unlisted eve crowds" + (" (Demera bonfire at Meskel Square)" if "Meskel" in eve_name[d] else "")
    if len(ids) > 1:
        return f"{len(ids)} events at once ({', '.join(ev.set_index('event_id').loc[ids, 'event_type'])}): features only see the largest"
    if ids:
        return f"inside {ev.set_index('event_id').loc[ids[0], 'event_type']} window: crowd bigger than its attendance suggests"
    if r.rain_3h > 0:
        return f"rain ({r.rain_3h:.1f} mm in 3 h) on a forecast-sensitive hour"
    return "no listed event or rain: likely an unlisted event"
top10 = oof.sort_values("abs_err", ascending=False).head(10).copy()
top10["hypothesis"] = top10.apply(hypothesis, axis=1)
top10 = top10[["zone", "pickup_hour", "trips", "pred", "err", "rain_3h", "ev_ids", "hypothesis"]]
R.h("D7 Error analysis")
R.p(f"Out-of-fold predictions of the tuned model over all {len(CUTS)} rolling folds ({len(oof):,} zone-hours, "
    f"{oof.pickup_hour.min():%d %b} – {oof.pickup_hour.max():%d %b}), so holidays are included.")
R.p("**(a) By zone**"); R.table(by_zone.reset_index(), floatfmt=".2f")
R.p("**By hour of day**"); R.table(by_hour.reset_index(), floatfmt=".2f")
R.p("**By day type**"); R.table(by_day.reset_index(), floatfmt=".2f")
R.p("**(b) The 10 largest errors**"); R.table(top10, floatfmt=".1f")
n_eve = top10.hypothesis.str.startswith("eve of").sum()
R.p(f"Errors scale with volume: Kazanchis, Merkato and Bole have the largest RMSE but similar errors relative to their demand, and the "
    f"evening peak hours carry the most error. Holidays are harder than ordinary days. **{n_eve} of the 10 biggest misses are on the eve of a "
    "public holiday** — Meskel eve (the Demera bonfire at Meskel Square, next to Kazanchis and Piassa) and Ethiopian New Year eve "
    "(Merkato shopping) — which the calendar does not list. The remaining one has no listed event or rain, so it is "
    "probably an unlisted gathering. All 10 are under-forecasts (positive errors): the model plays safe on surges.")
by_zone

,zone_hours,mean_trips,rmse,mae,bias(actual-pred),rmse_pct_of_mean
zone,,,,,,
Kazanchis,1641.0,37.615478,13.406843,6.942706,1.094348,35.641826
Merkato,1634.0,45.929621,11.295465,7.346636,0.879659,24.592986
Bole,1638.0,44.771673,10.637152,7.750660,0.712496,23.758665
Megenagna,1625.0,44.014769,10.104281,7.008039,0.783666,22.956569
Piassa,1642.0,32.588002,8.705071,5.619473,0.745957,26.712503
Lideta,1633.0,32.357930,7.804822,5.633655,0.703975,24.120276
CMC,1628.0,29.819717,7.248778,5.294456,0.700485,24.308673
Gerji,1631.0,28.263335,6.888896,4.986560,0.887061,24.373965
Kolfe,1643.0,26.403530,6.810235,4.898846,0.327177,25.792894


## D7 (continued) — Demand-level confusion matrix

In [10]:
# Trips are a count, so a classic confusion matrix needs classes. We give every zone-hour a demand level relative to
# that zone's own recent history: thresholds are the 25th / 75th / 90th percentiles of the zone's trips over the
# last 8 weeks of each fold's TRAINING data (nothing fitted on the fold being scored). The same thresholds turn the
# actual count, the model's forecast and the seasonal-naive forecast into Quiet / Normal / Busy / Peak.
from src.predict import calibrate
DEMAND_LEVELS = ["Quiet", "Normal", "Busy", "Peak"]
LEVEL_QS = [0.25, 0.75, 0.90]
_, q_int, _ = calibrate(m, BASE, best)
q80_hi = q_int["80"][1]
def to_level(zones, vals, th):
    return (np.asarray(vals, dtype=float)[:, None] >= th.loc[zones].to_numpy()).sum(axis=1)
lv = []
for c in CUTS:
    a, b = T.split(m, c)
    f = oof[oof.fold == c]
    recent = a[a.pickup_hour >= a.pickup_hour.max() - pd.Timedelta(weeks=8)]
    th = recent.groupby("zone")["trips"].quantile(LEVEL_QS).unstack()
    upper80 = f.pred.values + q80_hi * np.sqrt(f.pred.values + 1)
    lv.append(pd.DataFrame({"zone": f.zone.values, "fold": c,
                            "actual": to_level(f.zone, f.trips, th),
                            "model": to_level(f.zone, f.pred, th),
                            "seasonal_naive": to_level(b.zone, T.seasonal_naive(a, b, 4), th),
                            "model_upper80": to_level(f.zone, upper80, th)}))
lv = pd.concat(lv, ignore_index=True)

def confusion(col):
    M = np.zeros((4, 4), dtype=int)
    np.add.at(M, (lv.actual.values, lv[col].values), 1)
    return M
CM = {col: confusion(col) for col in ["model", "seasonal_naive", "model_upper80"]}
def level_metrics(col):
    M, y, p = CM[col], lv.actual, lv[col]
    rec = np.diag(M) / M.sum(axis=1); prec = np.diag(M) / np.maximum(M.sum(axis=0), 1)
    f1 = np.where(rec + prec > 0, 2 * rec * prec / (rec + prec), 0)
    return {"exact_level_accuracy": (y == p).mean(), "within_one_level": ((y - p).abs() <= 1).mean(),
            "macro_f1": f1.mean(), "peak_recall": rec[3], "peak_precision": prec[3],
            "peaks_called_quiet_or_normal": int(M[3, :2].sum()), "under_called_share": (p < y).mean(),
            "over_called_share": (p > y).mean()}
cm_summary = pd.DataFrame({"LightGBM (tuned)": level_metrics("model"),
                           "Seasonal naive (4 weeks)": level_metrics("seasonal_naive"),
                           "LightGBM, upper 80% bound": level_metrics("model_upper80")}).T
def cm_table(M):
    pct = M / M.sum(axis=1, keepdims=True)
    t = pd.DataFrame([[f"{M[i, j]:,} ({pct[i, j]:.0%})" for j in range(4)] for i in range(4)],
                     columns=[f"pred {l}" for l in DEMAND_LEVELS])
    t.insert(0, "actual", DEMAND_LEVELS); t["total"] = [f"{n:,}" for n in M.sum(axis=1)]
    return t

fig, axes = plt.subplots(1, 3, figsize=(19, 6.2))
titles = {"model": "LightGBM forecast", "seasonal_naive": "Seasonal-naive baseline",
          "model_upper80": "LightGBM upper 80% bound (peak alert)"}
for ax, (col, title) in zip(axes, titles.items()):
    M = CM[col]; pct = M / M.sum(axis=1, keepdims=True)
    ax.imshow(pct, cmap=S.SEQ_CMAP, vmin=0, vmax=1)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f"{pct[i, j]:.0%}\n{M[i, j]:,}", ha="center", va="center", fontsize=11,
                    color="white" if pct[i, j] > 0.55 else S.INK, fontweight="bold" if i == j else "normal")
    ax.set_xticks(range(4), DEMAND_LEVELS); ax.set_yticks(range(4), DEMAND_LEVELS)
    ax.set_xlabel("Predicted level"); ax.set_ylabel("Actual level" if col == "model" else "")
    ax.grid(False); ax.spines[:].set_visible(False)
    acc = (lv.actual == lv[col]).mean()
    ax.set_title(f"{title}\nexact {acc:.0%} · peak recall {M[3, 3] / M[3].sum():.0%}", fontsize=12)
fig.suptitle("D7  Demand-level confusion matrices, 5 rolling folds (row % = share of each actual level)",
             fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(config.REPORTS / "d7_confusion_matrix.png", dpi=150, bbox_inches="tight"); plt.close(fig);

mc, mn, mu = cm_summary.iloc[0], cm_summary.iloc[1], cm_summary.iloc[2]
R.h("D7 (continued) Demand-level confusion matrix", 3)
R.p("Trips are a count, so we turn them into four demand levels to read the model like a classifier. For every fold, each zone's "
    "thresholds are the 25th, 75th and 90th percentiles of its trips over the last 8 weeks of that fold's training data: "
    "**Quiet** (below the 25th), **Normal**, **Busy** (75th–90th) and **Peak** (above the 90th). The same thresholds are applied to the "
    f"actual count and to each forecast, over the same {len(lv):,} out-of-fold zone-hours as above.")
R.p("**LightGBM (tuned): rows = actual level, columns = predicted level (row %)**"); R.table(cm_table(CM["model"]))
R.p("**Seasonal-naive baseline (same thresholds)**"); R.table(cm_table(CM["seasonal_naive"]))
R.p("**Summary**"); R.table(cm_summary.reset_index().rename(columns={"index": "forecast"}), floatfmt=".3f")
R.img("d7_confusion_matrix.png", "Demand-level confusion matrices")
R.p(f"The model puts {mc.exact_level_accuracy:.0%} of zone-hours in the right level (baseline {mn.exact_level_accuracy:.0%}) and "
    f"{mc.within_one_level:.1%} within one level, with a macro-F1 of {mc.macro_f1:.2f} against {mn.macro_f1:.2f}. Its weak spot is the "
    f"top of the range: it catches only {mc.peak_recall:.0%} of actual peak hours, and most of the misses are called Busy rather than "
    f"Quiet or Normal ({mc.peaks_called_quiet_or_normal:,} peaks were called Quiet or Normal). That matches D7: the model under-forecasts surges. "
    f"For planning, using the **upper bound of the 80% interval** as a peak alert raises peak recall to {mu.peak_recall:.0%}, at the "
    f"cost of precision ({mu.peak_precision:.0%} of alerts are real peaks), which is the right trade when a missed peak means "
    "riders without cars. (The interval width is calibrated on the last two folds, so the alert row is slightly optimistic for those folds.)")
cm_summary

,exact_level_accuracy,within_one_level,macro_f1,peak_recall,peak_precision,peaks_called_quiet_or_normal,under_called_share,over_called_share
LightGBM (tuned),0.778892,0.994298,0.732132,0.574776,0.774842,81.0,0.133693,0.087415
Seasonal naive (4 weeks),0.731697,0.979075,0.669971,0.484022,0.624176,248.0,0.154414,0.113889
"LightGBM, upper 80% bound",0.628653,0.971999,0.582068,0.923732,0.496564,5.0,0.019448,0.351899


## D8 — Response to findings

In [11]:
roll_final = T.rolling_origin(m, CUTS, lambda a, b: T.fit_predict("lightgbm", a, b, FINAL, best)[1])
oof8 = []
for c in CUTS:
    a, b = T.split(m, c)
    oof8.append(b.assign(pred=T.fit_predict("lightgbm", a, b, FINAL, best)[1]))
oof8 = pd.concat(oof8, ignore_index=True)
eve_mask = oof8.is_holiday_eve == 1
_, p_fin, _ = T.fit_predict("lightgbm", tr, va, FINAL, best)
d8 = pd.DataFrame([
    {"model": "tuned, before D8", "rolling_rmse_mean": roll_tun.rmse.mean(), "rolling_mae_mean": roll_tun.mae.mean(),
     "rmse_on_holiday_eves": np.sqrt(((oof.trips - oof.pred)[oof8.is_holiday_eve.values == 1] ** 2).mean()),
     "main_split_rmse": T.rmse(va.trips, p_tun)},
    {"model": "tuned + is_holiday_eve (final)", "rolling_rmse_mean": roll_final.rmse.mean(), "rolling_mae_mean": roll_final.mae.mean(),
     "rmse_on_holiday_eves": np.sqrt(((oof8.trips - oof8.pred)[eve_mask] ** 2).mean()), "main_split_rmse": T.rmse(va.trips, p_fin)}])
fold_cmp = pd.DataFrame({"cut": roll_tun.cut, "before": roll_tun.rmse, "after": roll_final.rmse})
R.h("D8 Response to findings")
R.p("Change: D7 showed the largest misses on holiday eves, so we added `is_holiday_eve` (1 on the day before any confirmed public "
    "holiday, all zones; the tree learns which zones react). We also tried a count of simultaneous events, which did not help and was dropped.")
R.table(d8, floatfmt=".3f"); R.table(fold_cmp, floatfmt=".3f")
gain = d8.rolling_rmse_mean[0] - d8.rolling_rmse_mean[1]
R.p(f"It helped where it should: RMSE on holiday-eve hours fell from {d8.rmse_on_holiday_eves[0]:.1f} to {d8.rmse_on_holiday_eves[1]:.1f}, "
    f"and the rolling RMSE moved by {-gain:+.3f} overall. The main-split fortnight (18–31 Oct) has no holiday eve, so its score moves only by "
    f"noise ({d8.main_split_rmse[0]:.3f} -> {d8.main_split_rmse[1]:.3f}). We keep the feature: it costs nothing and protects the eves of "
    "future holidays. The two-events-at-once misses remain (too few examples to learn from).")
d8

,model,rolling_rmse_mean,rolling_mae_mean,rmse_on_holiday_eves,main_split_rmse
0,"tuned, before D8",8.732730,5.734509,16.893665,8.137937
1,tuned + is_holiday_eve (final),8.680175,5.727557,16.111117,8.090693


## D9 — Plain-language metric

In [12]:
rm, ma, mean_t = T.rmse(va.trips, p_fin), T.mae(va.trips, p_fin), va.trips.mean()
drivers_tph = float((T.trainable(m).trips / T.trainable(m).active_drivers).median())
d9 = pd.DataFrame([{"metric": "RMSE", "trips per zone-hour": rm, "% of mean demand": 100 * rm / mean_t, "drivers (÷ 1.3)": rm / config.TRIPS_PER_DRIVER_HOUR},
                   {"metric": "MAE", "trips per zone-hour": ma, "% of mean demand": 100 * ma / mean_t, "drivers (÷ 1.3)": ma / config.TRIPS_PER_DRIVER_HOUR}])
R.h("D9 Plain-language metric")
R.table(d9, floatfmt=".1f")
R.p(f"**For an operations manager:** in a typical zone and hour (about {mean_t:.0f} trip requests), our forecast is off by about "
    f"**{ma:.1f} trips** on average (MAE, {100*ma/mean_t:.0f}% of demand); bigger misses pull the root-mean-square error up to {rm:.1f} trips "
    f"({100*rm/mean_t:.0f}%). With drivers completing about {config.TRIPS_PER_DRIVER_HOUR} trips per hour (history median {drivers_tph:.2f}), "
    f"that is roughly **{ma/config.TRIPS_PER_DRIVER_HOUR:.0f} drivers too many or too few per zone per hour** on a normal hour, and around "
    f"{rm/config.TRIPS_PER_DRIVER_HOUR:.0f} when the occasional big misses (holiday eves, surges) are weighted in (RMSE). The seasonal-naive rule of thumb would be off by "
    f"{d1.loc[2,'mae']:.1f} trips ({d1.loc[2,'mae']/config.TRIPS_PER_DRIVER_HOUR:.0f} drivers).")
d9

,metric,trips per zone-hour,% of mean demand,drivers (÷ 1.3)
0,RMSE,8.090693,24.189050,6.223610
1,MAE,5.601771,16.747824,4.309054


## Final model (trained on all of 1 Jan – 31 Oct) and saved for the demo

In [13]:
full = T.trainable(m)
final_model = T.make_model("lightgbm", FINAL, best)
t0 = time.perf_counter(); final_model.fit(full[FINAL], full.trips); fit_s = time.perf_counter() - t0
zone_avg_fare = full.groupby("zone")["avg_fare_birr"].median()
profile_typical = full.groupby(["zone", "dow", "hour"])["trips"].mean().rename("typical").reset_index()
bundle = {"model": final_model, "features": FINAL, "params": {**T.LGB_PARAMS, **best}, "team": config.TEAM_NAME,
          "trained_on": f"{full.pickup_hour.min():%Y-%m-%d} to {full.pickup_hour.max():%Y-%m-%d}", "n_train": len(full),
          "validation": {"main_split_rmse": rm, "main_split_mae": ma, "rolling_rmse_mean": roll_final.rmse.mean(),
                         "rolling_rmse_sd": roll_final.rmse.std(), "seasonal_naive_rmse": SN},
          "zone_avg_fare": zone_avg_fare.to_dict()}
config.MODELS.mkdir(exist_ok=True)
joblib.dump(bundle, config.MODELS / "final_model.joblib", compress=3)
(config.MODELS / "final_model_params.json").write_text(pd.Series(bundle["params"]).to_json(indent=2))
print(f"fitted on {len(full):,} rows in {fit_s:.1f}s; saved models/final_model.joblib")
R.h("Final model")
R.p(f"LightGBM (Poisson), tuned parameters, {len(FINAL)} features, refit on all {len(full):,} observed zone-hours of 1 Jan – 31 Oct and saved "
    f"to `models/final_model.joblib` (with its feature list, parameters and validation scores). **Validation score: RMSE {rm:.2f}, MAE {ma:.2f} "
    f"on 18–31 Oct; rolling-origin RMSE {roll_final.rmse.mean():.2f} ± {roll_final.rmse.std():.2f}** (seasonal naive: {SN:.2f} on 18–31 Oct, {roll_sn.rmse.mean():.2f} rolling). "
    "Uses weather (6) and event (12 + holiday flags) features (Rule 5), only forecast-time features (Rule 6), chronological validation only "
    "(Rule 7) and nothing fitted on test (Rule 8).")

fitted on 83,104 rows in 2.2s; saved models/final_model.joblib


## fig10 — Model comparison

In [14]:
bars = [("Mean\npredictor", d1.loc[0, "rmse"], None, S.NEUTRAL),
        ("Seasonal naive\n(whole period)", d1.loc[1, "rmse"], None, S.NEUTRAL),
        ("Seasonal naive\n(last 4 weeks)", d1.loc[2, "rmse"], roll_sn.rmse, S.NEUTRAL),
        ("Moving avg\n(last 7 days)", d1.loc[3, "rmse"], ROLL["moving_average"].rmse, S.NEUTRAL)]
for name, lab in [("arima", "ARIMA\n(per zone)"), ("ridge", "Ridge"), ("random_forest", "Random\nforest"),
                  ("gradient_boosting", "Gradient\nboosting"), ("hist_gb", "HistGradient-\nBoosting"), ("lightgbm", "LightGBM\n(default)")]:
    bars.append((lab, T.rmse(va.trips, PRED_MAIN[name]), ROLL[name].rmse, S.SERIES[0]))
bars.append(("LightGBM\ntuned", T.rmse(va.trips, p_tun), roll_tun.rmse, S.SERIES[0]))
bars.append(("FINAL\ntuned + eve", rm, roll_final.rmse, S.SERIES[1]))
fig, ax = plt.subplots(figsize=(19, 6.5))
for i, (lab, v, roll, col) in enumerate(bars):
    ax.bar(i, v, color=col, width=0.65)
    ax.text(i - 0.12, v + 0.4, f"{v:.2f}", ha="center", fontsize=10)   # left of the diamond
    if roll is not None:
        ax.errorbar(i + 0.22, roll.mean(), yerr=roll.std(), fmt="D", color=S.INK, ms=6, capsize=5, lw=1.5)
ax.errorbar([], [], yerr=[], fmt="D", color=S.INK, ms=6, capsize=5, label=f"rolling origin, {len(CUTS)} folds: mean ± sd")
ax.axhline(d1.loc[2, "rmse"], color=S.NEUTRAL, ls="--", lw=1.2, label="best baseline (seasonal naive, last 4 weeks)")
ax.plot([], [], color=S.NEUTRAL, lw=8, label="baselines"); ax.plot([], [], color=S.SERIES[0], lw=8, label="models")
ax.plot([], [], color=S.SERIES[1], lw=8, label="final model")
ax.set_xticks(range(len(bars))); ax.set_xticklabels([b[0] for b in bars], fontsize=10)
ax.set_ylabel("validation RMSE (trips per zone-hour)"); ax.set_ylim(0, d1.loc[0, "rmse"] * 1.1)
ax.legend(loc="upper right"); ax.grid(axis="x", visible=False)
ax.set_title("fig10  Validation RMSE, 18-31 Oct 2025 (bars) and rolling-origin spread (diamonds)")
S.save(fig, "fig10_model_comparison.png");

saved fig10_model_comparison.png (2850x975 px at 150 dpi)


## fig11 — Forecast vs actual

In [15]:
vv = va.assign(pred=p_fin)
zones3 = ["Kazanchis", "Bole", "Merkato"]
fig, axes = plt.subplots(3, 1, figsize=(16, 11), sharex=True)
for ax, z in zip(axes, zones3):
    d = vv[vv.zone == z].sort_values("pickup_hour")
    full_idx = pd.date_range(va.pickup_hour.min(), va.pickup_hour.max(), freq="h")
    a = d.set_index("pickup_hour").reindex(full_idx)
    ax.plot(a.index, a.trips, color=S.INK, lw=1.4, label="actual")
    ax.plot(a.index, a.pred, color=S.SERIES[1], lw=1.6, label="forecast (model trained to 17 Oct)")
    ax.set_title(f"{z}: RMSE {T.rmse(d.trips, d.pred):.1f}, MAE {T.mae(d.trips, d.pred):.1f} trips (mean {d.trips.mean():.0f})", fontsize=12, loc="left")
    ax.set_ylabel("trips per hour"); ax.set_ylim(0, None)
axes[0].legend(loc="upper right", ncol=2)
axes[-1].set_xlabel("date and hour (EAT), 18-31 Oct 2025")
fig.suptitle("fig11  Hourly forecast vs actual over the validation fortnight", fontsize=15, fontweight="bold")
fig.tight_layout()
S.save(fig, "fig11_forecast_vs_actual.png");

saved fig11_forecast_vs_actual.png (2400x1650 px at 150 dpi)


## fig12 — Feature importance

In [16]:
m_tr = T.make_model("lightgbm", FINAL, best).fit(tr[FINAL], tr.trips)
pi = permutation_importance(m_tr, va[FINAL], va.trips, n_repeats=5, random_state=config.RANDOM_STATE,
                            scoring="neg_root_mean_squared_error")
imp = pd.DataFrame({"feature": FINAL, "rmse_increase": pi.importances_mean, "sd": pi.importances_std}).sort_values("rmse_increase", ascending=False)
fam = {f: g for g, fs in [("calendar", features.CALENDAR_FEATURES + features.D8_FEATURES), ("zone", features.ZONE_FEATURES),
                          ("weather", features.WEATHER_FEATURES), ("events", features.EVENT_FEATURES), ("lag / trend", features.LAG_FEATURES)] for f in fs}
imp["family"] = imp.feature.map(fam)
top = imp.head(18).iloc[::-1]
colors = {"weather": S.SERIES[0], "events": S.SERIES[1], "calendar": S.NEUTRAL, "zone": S.NEUTRAL, "lag / trend": S.NEUTRAL}
fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(top.feature, top.rmse_increase, xerr=top.sd, color=[colors[f] for f in top.family], height=0.65,
        error_kw=dict(ecolor=S.INK_2, lw=1, capsize=2))
ax.set_xlabel("increase in validation RMSE when the feature is shuffled (trips)")
ax.plot([], [], color=S.SERIES[0], lw=8, label="weather feature"); ax.plot([], [], color=S.SERIES[1], lw=8, label="event feature")
ax.plot([], [], color=S.NEUTRAL, lw=8, label="calendar / zone / lag")
ax.legend(loc="lower right"); ax.grid(axis="y", visible=False)
xmax = 1.6 * top.rmse_increase.iloc[:-1].max()   # the profile bar dwarfs the rest: clip it and print its value
ax.set_xlim(0, xmax)
ax.annotate(f"{top.feature.iloc[-1]}: {top.rmse_increase.iloc[-1]:.1f}  (bar cut off) →", xy=(xmax * 0.98, len(top) - 1),
            ha="right", va="center", color="white", fontsize=10, fontweight="bold")
ax.set_title("fig12  Permutation importance, top 18 features (18-31 Oct validation)")
S.save(fig, "fig12_feature_importance.png");
imp.to_csv(config.REPORTS / "D_permutation_importance.csv", index=False)
w_rank = imp.reset_index(drop=True).query("family == 'weather'").index.min() + 1
e_rank = imp.reset_index(drop=True).query("family == 'events'").index.min() + 1

saved fig12_feature_importance.png (1800x1200 px at 150 dpi)


## Captions for fig10–12 and the report

In [17]:
CAP = {
 "fig10_model_comparison.png":
   f"Validation RMSE on 18-31 Oct for the baselines (mean, seasonal naive, moving average), six model families (ARIMA, ridge, random forest and three gradient-boosting variants) and the tuned/final LightGBM (bars, from zero), with the "
   f"rolling-origin mean ± sd over {len(CUTS)} folds (diamonds). The final model ({rm:.2f}) beats the best baseline ({SN:.2f}) by "
   f"{100*(1-rm/SN):.0f}% and does so in every fold. So what: boosted trees on well-joined features are the right tool here; tuning adds little.",
 "fig11_forecast_vs_actual.png":
   "Hourly forecasts (model trained up to 17 Oct) against actual trips for three contrasting zones over the full validation fortnight. "
   "The model follows the daily and weekly shape in each zone, including Kazanchis' empty weekends and Bole's weekend nights; the misses "
   "are the tallest surge peaks, which it under-shoots. So what: plan for the forecast, keep a buffer at known peaks (see the stretch intervals).",
 "fig12_feature_importance.png":
   f"Permutation importance: how much the validation RMSE rises when each feature is shuffled; weather features in blue, event features in "
   f"orange. The zone x weekday x hour profile dominates (its bar is cut off; it already encodes zone and hour, so those rank lower on their own), but the top weather feature ranks #{w_rank} and the top event feature #{e_rank}. "
   "So what: the joins add real, if smaller, signal on top of the calendar.",
}
cap_path = config.FIGURES / "figure_captions.md"
text = cap_path.read_text(encoding="utf-8").split("\n## fig10_")[0].rstrip() + "\n"
for name, t in CAP.items():
    text += f"\n## {name}\n\n{t}\n"
cap_path.write_text(text, encoding="utf-8")
print(text.count("## fig"), "captions in figure_captions.md")
R.h("Figures")
for name in CAP:
    R.img(f"../figures/{name}", name); R.p(CAP[name])
R.write(config.REPORTS / "D_model_evaluation.md")

12 captions in figure_captions.md
wrote D_model_evaluation.md (34,752 chars)
